In [ ]:
"""
Hopfield Network Associative Memory & Pattern Reconstruction (handwritten digits)
--------------------------------------------------------------------------------
Dataset : Optical Recognition of Handwritten Digits (UCI optdigits, 8x8, values 0-16)

Run     : python hopfield_digits.py [optdigits.tra | optdigits.tes]

Data lookup order: file argument -> optdigits.tra/.tes in folder ->
                   scikit-learn's bundled UCI optdigits copy (offline) -> UCI download.
Pixels >= 8 become +1 (active), others -1 (inactive).  64 neurons.
Learning  : Hebb rule  W = (1/N) sum_p  x_p x_p^T ,  zero diagonal
Update    : asynchronous, random order, until no neuron changes
Energy    : E = -1/2 x^T W x

Requires: numpy   (scikit-learn optional, used as an offline data source)
"""
import os
import sys
import urllib.request

import numpy as np

SEED = 5
DEFAULT_DIGITS = [2, 6, 7]     # weakly correlated digits: plain Hebb stores these reliably
N = 64
THRESHOLD = 8
UCI_BASE = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/"


# ----------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------
def _csv(path):
    a = np.loadtxt(path, delimiter=",")
    return a[:, :64], a[:, 64].astype(int)


def load_data(path=None):
    for p in ([path] if path else []) + ["optdigits.tra", "optdigits.tes"]:
        if p and os.path.exists(p):
            print(f"[data] Loading {p}")
            X, y = _csv(p); return X, y, "file: " + p
    try:
        from sklearn.datasets import load_digits
        d = load_digits()
        print("[data] Using scikit-learn bundled UCI optdigits (1797 samples).")
        return d.data.astype(float), d.target.astype(int), "UCI optdigits via scikit-learn"
    except Exception:
        pass
    try:
        print("[data] Downloading optdigits.tra from UCI ...")
        urllib.request.urlretrieve(UCI_BASE + "optdigits.tra", "optdigits.tra")
        X, y = _csv("optdigits.tra"); return X, y, "UCI download"
    except Exception as e:
        sys.exit(f"[data] Could not load a dataset ({e}). Put optdigits.tra next to this script.")


def to_bipolar(X): return np.where(X >= THRESHOLD, 1, -1).astype(int)


def show(p, title=""):
    if title: print(title)
    for r in p.reshape(8, 8):
        print("   " + " ".join("#" if v > 0 else ("." if v < 0 else " ") for v in r))


def row(p):
    return [" ".join("#" if v > 0 else ("." if v < 0 else " ") for v in r) for r in p.reshape(8, 8)]


def show_many(pats, titles, gap="     "):
    print("   " + gap.join(f"{t:<15}" for t in titles))
    rows = [row(p) for p in pats]
    for r in range(8):
        print("   " + gap.join(f"{rw[r]:<15}" for rw in rows))


# ----------------------------------------------------------------------------
# Hopfield network
# ----------------------------------------------------------------------------
class Hopfield:
    def __init__(self):
        self.W = np.zeros((N, N)); self.P = np.zeros((0, N), int); self.labels = []; self.rule = "hebb"

    def store(self, patterns, labels, rule="hebb"):
        """rule 'hebb': W = (1/N) sum x x^T   |   'pinv': projection (pseudo-inverse) rule, for comparison"""
        self.P = np.array(patterns, int); self.labels = list(labels); self.rule = rule
        if rule == "hebb":
            self.W = sum(np.outer(p, p) for p in self.P).astype(float) / N
        else:
            self.W = self.P.T @ np.linalg.pinv(self.P @ self.P.T) @ self.P
        np.fill_diagonal(self.W, 0)

    def energy(self, x): return float(-0.5 * x @ self.W @ x)

    def recall(self, x, rng, max_sweeps=50):
        """Asynchronous recall. x may contain 0 (unknown). Returns state, sweeps, energy trace."""
        x = x.astype(int).copy()
        trace = [self.energy(x)]
        for sw in range(1, max_sweeps + 1):
            changed = 0
            for i in rng.permutation(N):
                h = self.W[i] @ x
                new = x[i] if h == 0 and x[i] != 0 else (1 if h >= 0 else -1)
                if new != x[i]: x[i] = new; changed += 1
            trace.append(self.energy(x))
            if changed == 0: return x, sw, trace
        return x, max_sweeps, trace

    def identify(self, x):
        """Classify a fixed point: ('stored', label) | ('mirror', label) | ('spurious', nearest_label, hamming)."""
        for p, l in zip(self.P, self.labels):
            if np.array_equal(x, p): return "stored", l
        for p, l in zip(self.P, self.labels):
            if np.array_equal(x, -p): return "mirror", l
        d = [(int((x != p).sum()), l) for p, l in zip(self.P, self.labels)]
        h, l = min(d)
        return "spurious", l, h


# ----------------------------------------------------------------------------
# application
# ----------------------------------------------------------------------------
def parse_digits(s, default):
    try:
        v = sorted({int(t) for t in s.replace(",", " ").split() if 0 <= int(t) <= 9})
        return v or default
    except ValueError:
        return default


class App:
    def __init__(self, X, y, source):
        self.source = source
        self.rng = np.random.default_rng(SEED)
        self.Xb, self.y = to_bipolar(X), y
        self.net = Hopfield()
        self.proto = {d: np.where(self.Xb[y == d].mean(0) > 0, 1, -1) for d in range(10)}
        self.digits = []
        self.last = {}
        self.store_digits(DEFAULT_DIGITS, quiet=True)

    def store_digits(self, digits, quiet=False, mode="prototype", rule="hebb"):
        self.digits = list(digits)
        pats = []
        for d in self.digits:
            if mode == "prototype": pats.append(self.proto[d])
            else:
                idx = np.where(self.y == d)[0]; pats.append(self.Xb[self.rng.choice(idx)])
        self.net.store(pats, self.digits, rule)
        self.mode = mode; self.rule = rule

    def need(self):
        if len(self.digits) == 0: self.op1()

    def stable_count(self):
        return sum(np.array_equal(np.where(self.net.W @ p >= 0, 1, -1), p) for p in self.net.P)

    def noisy(self, x, frac):
        x = x.copy(); frac = min(max(float(frac), 0.0), 1.0)
        k = int(round(frac * N))
        x[self.rng.choice(N, k, replace=False)] *= -1
        return x

    def pick_digit(self):
        while True:
            s = input(f"Stored digit to use {self.digits}: ").strip()
            if s.isdigit() and int(s) in self.digits: return int(s)
            if s == "": return int(self.rng.choice(self.digits))
            print("  that digit is not stored.")

    # -- 1
    def op1(self):
        s = input("Digits to store (e.g. '2 6 7' or '0 1 2 3 4 5 6 7 8 9', default 2 6 7): ").strip()
        digits = parse_digits(s, DEFAULT_DIGITS)
        m = input("Pattern source: 1) class-average prototype (default)  2) one random sample each: ").strip()
        r = input("Learning rule: 1) Hebbian (default)  2) pseudo-inverse (comparison, handles correlated digits): ").strip()
        self.store_digits(digits, mode="sample" if m == "2" else "prototype", rule="pinv" if r == "2" else "hebb")
        rn = "Hebb rule  W = (1/%d) sum x x^T" % N if self.rule == "hebb" else "pseudo-inverse rule  W = P^T (P P^T)^-1 P"
        print(f"\nStored {len(digits)} patterns by the {rn}  ({self.mode})")
        for d, p in zip(self.digits, self.net.P):
            show(p, f"\nDigit {d}  (energy {self.net.energy(p):.3f})")
        print(f"\nTheoretical capacity of {N} neurons ~ 0.138*N = {0.138*N:.1f} random patterns")
        st = self.stable_count()
        print(f"Stored patterns that are stable fixed points: {st}/{len(digits)}")
        if st < len(digits) and self.rule == "hebb":
            print("-> Digit images are highly correlated, so plain Hebbian crosstalk destabilises some memories."
                  "\n   Store fewer / less similar digits (e.g. 2 6 7) or compare with the pseudo-inverse rule.")
        c = np.corrcoef(self.net.P)
        off = np.abs(c[~np.eye(len(digits), dtype=bool)]) if len(digits) > 1 else np.array([0])
        print(f"Mean |correlation| between stored patterns: {off.mean():.2f} "
              f"(high correlation lowers real capacity)")

    # -- 2
    def op2(self):
        self.need()
        d = self.pick_digit()
        f = float(input("Noise fraction (0-1, default 0.2): ") or 0.2)
        true = self.net.P[self.digits.index(d)]
        x0 = self.noisy(true, f)
        x, sw, tr = self.net.recall(x0, self.rng)
        show_many([x0, x, true], ["Noisy input", "Recalled", "Original"])
        res = self.net.identify(x)
        print(f"\nPixels corrupted: {(x0 != true).sum()}   remaining wrong: {(x != true).sum()}   sweeps: {sw}")
        print("Energy per sweep: " + " -> ".join(f"{e:.2f}" for e in tr))
        self.report_fp(res, d)

    def report_fp(self, res, d):
        if res[0] == "stored":
            print(f"Result: converged to stored digit {res[1]}  -> {'CORRECT' if res[1]==d else 'WRONG MEMORY'}")
        elif res[0] == "mirror":
            print(f"Result: converged to the INVERSE (mirror image) of digit {res[1]} - a false memory")
        else:
            print(f"Result: SPURIOUS state (closest to digit {res[1]}, {res[2]} pixels away) - a false memory")

    # -- 3
    def op3(self):
        self.need()
        print("Missing region: 1) top half 2) bottom half 3) left half 4) right half 5) random pixels")
        m = int(input("Mode (default 2): ") or 2)
        keep = np.ones((8, 8), bool)
        if m == 1: keep[:4] = False
        elif m == 2: keep[4:] = False
        elif m == 3: keep[:, :4] = False
        elif m == 4: keep[:, 4:] = False
        else:
            f = float(input("Fraction of pixels missing (default 0.5): ") or 0.5)
            keep = (self.rng.permutation(64) >= int(f * 64)).reshape(8, 8)
        keep = keep.ravel()
        d = self.pick_digit(); true = self.net.P[self.digits.index(d)]
        x0 = np.where(keep, true, 0)
        x, sw, tr = self.net.recall(x0, self.rng)
        show_many([x0, x, true], ["Input (blank=?)", "Reconstructed", "Original"])
        hid = ~keep
        print(f"\nMissing pixels: {hid.sum()}   restored correctly: {(x[hid]==true[hid]).sum()} "
              f"({(x[hid]==true[hid]).mean()*100:.1f}%)   sweeps: {sw}")
        self.report_fp(self.net.identify(x), d)
        ok = 0
        for dd, p in zip(self.digits, self.net.P):
            xx, _, _ = self.net.recall(np.where(keep, p, 0), self.rng)
            ok += np.array_equal(xx, p)
        print(f"Same mask on every stored digit: {ok}/{len(self.digits)} perfectly reconstructed")

    # -- 4
    def op4(self):
        self.need()
        trials = int(input("Trials per digit and noise level (default 30): ") or 30)
        levels = [0, .05, .1, .15, .2, .25, .3, .4, .5]
        print(f"\nPerfect-recall rate (%) per stored digit:")
        print(f"{'noise':>7}" + "".join(f"{d:>7}" for d in self.digits) + f"{'mean':>8}")
        for f in levels:
            rates = []
            for p in self.net.P:
                ok = sum(np.array_equal(self.net.recall(self.noisy(p, f), self.rng)[0], p) for _ in range(trials))
                rates.append(ok / trials * 100)
            print(f"{f*100:>6.0f}%" + "".join(f"{r:>7.0f}" for r in rates) + f"{np.mean(rates):>8.1f}")
        print("\nTolerance = highest noise level with mean recall >= 90% (see table). "
              "Above ~25-30% the input is closer to another attractor, or to the mirror image.")

    # -- 5
    def op5(self):
        trials = int(input("Trials per size (default 20): ") or 20)
        print("\nA) Digit patterns (random real samples of all digits stored together)")
        print(f"{'stored':>7}{'stable %':>10}{'recall@10% noise':>18}{'recall@20% noise':>18}{'pinv stable %':>15}")
        for m in (1, 2, 3, 4, 5, 6, 8, 10, 15, 20):
            sw, r10, r20, pv = [], [], [], []
            for _ in range(trials):
                idx = self.rng.choice(len(self.y), m, replace=False)
                netp = Hopfield()
                if len({tuple(r) for r in self.Xb[idx]}) == m:
                    netp.store(self.Xb[idx], self.y[idx], "pinv")
                    pv.append(np.mean([np.array_equal(np.where(netp.W @ p >= 0, 1, -1), p) for p in netp.P]))
                net = Hopfield(); net.store(self.Xb[idx], self.y[idx])
                sw.append(np.mean([np.array_equal(np.where(net.W @ p >= 0, 1, -1), p) for p in net.P]))
                p = net.P[self.rng.integers(m)]
                r10.append(np.array_equal(net.recall(self.noisy(p, .1), self.rng)[0], p))
                r20.append(np.array_equal(net.recall(self.noisy(p, .2), self.rng)[0], p))
            print(f"{m:>7}{np.mean(sw)*100:>9.0f}%{np.mean(r10)*100:>17.0f}%{np.mean(r20)*100:>17.0f}%{np.mean(pv)*100 if pv else float('nan'):>14.0f}%")
        print("\nB) Random uncorrelated patterns (theory: capacity ~ 0.138 N = 8.8)")
        print(f"{'stored':>7}{'stable %':>10}")
        for m in (2, 4, 6, 8, 10, 14, 20):
            sw = []
            for _ in range(trials):
                P = self.rng.choice([-1, 1], (m, N)); net = Hopfield(); net.store(P, range(m))
                sw.append(np.mean([np.array_equal(np.where(net.W @ p >= 0, 1, -1), p) for p in P]))
            print(f"{m:>7}{np.mean(sw)*100:>9.0f}%")
        print("\nDigit images are strongly correlated, so their practical capacity is far below the random-pattern limit.")

    # -- 6
    def op6(self):
        self.need()
        n = int(input("Random starting states to probe (default 500): ") or 500)
        cnt = {"stored": 0, "mirror": 0, "spurious": 0}; examples = {}; spur = {}
        for _ in range(n):
            x, _, _ = self.net.recall(self.rng.choice([-1, 1], N), self.rng)
            r = self.net.identify(x); cnt[r[0]] += 1
            if r[0] != "stored":
                key = tuple(x)
                spur[key] = spur.get(key, 0) + 1; examples[key] = (x, r)
        print(f"\nFrom {n} random starts the network settled in:")
        print(f"  stored memories : {cnt['stored']:>4} ({cnt['stored']/n*100:.1f}%)")
        print(f"  mirror images   : {cnt['mirror']:>4} ({cnt['mirror']/n*100:.1f}%)   (-pattern is always an attractor)")
        print(f"  other spurious  : {cnt['spurious']:>4} ({cnt['spurious']/n*100:.1f}%)")
        print(f"  distinct false attractors found: {len(spur)}")
        for key, c in sorted(spur.items(), key=lambda t: -t[1])[:3]:
            x, r = examples[key]
            show(x, f"\nFalse memory reached {c}x - " +
                 (f"inverse of digit {r[1]}" if r[0] == 'mirror' else f"mix, nearest digit {r[1]} ({r[2]} px away)"))
            print(f"   energy {self.net.energy(x):.3f}  (stored patterns: "
                  + ", ".join(f"{d}:{self.net.energy(p):.2f}" for d, p in zip(self.digits, self.net.P)) + ")")
        # also: unstable stored patterns
        bad = [d for d, p in zip(self.digits, self.net.P) if not np.array_equal(np.where(self.net.W @ p >= 0, 1, -1), p)]
        if bad: print(f"\nWARNING: stored digits {bad} are not even fixed points (crosstalk).")

    # -- 7
    def op7(self):
        self.need()
        f = float(input("Noise on real handwritten probes (default 0.1): ") or 0.1)
        D = self.digits; k = len(D)
        for name, noisy_f in (("REAL unseen handwritten samples", 0.0), (f"real samples + {f*100:.0f}% noise", f)):
            C = np.zeros((k, k + 1), int)                         # last col = spurious/mirror
            for ti, d in enumerate(D):
                idx = np.where(self.y == d)[0]
                for i in self.rng.choice(idx, min(40, len(idx)), replace=False):
                    x0 = self.noisy(self.Xb[i], noisy_f) if noisy_f else self.Xb[i]
                    r = self.net.identify(self.net.recall(x0, self.rng)[0])
                    C[ti, D.index(r[1]) if r[0] == "stored" else k] += 1
            print(f"\n{name}\n   (rows = true digit, columns = recalled stored digit; 'sp' = spurious/mirror)")
            print("      " + "".join(f"{d:>5}" for d in D) + f"{'sp':>5}{'acc%':>7}")
            for ti, d in enumerate(D):
                print(f"  {d:>3} " + "".join(f"{C[ti,j]:>5}" for j in range(k + 1)) + f"{C[ti,ti]/C[ti].sum()*100:>7.0f}")
            pairs = sorted(((C[a, b], D[a], D[b]) for a in range(k) for b in range(k) if a != b and C[a, b]), reverse=True)[:3]
            if pairs: print("  Most confused: " + ", ".join(f"{a}->{b} ({n})" for n, a, b in pairs))
        ov = sorted(((int((self.net.P[a] == self.net.P[b]).sum()), D[a], D[b])
                     for a in range(k) for b in range(a + 1, k)), reverse=True)[:3]
        if ov: print("\nStored-pattern pairs with greatest pixel overlap: " + ", ".join(f"{a}/{b} ({n}/64)" for n, a, b in ov))

    # -- 8
    def op8(self):
        self.need()
        print("=" * 62); print("HOPFIELD RECALL PERFORMANCE REPORT"); print("=" * 62)
        print(f"Dataset   : {self.source}")
        print(f"Network   : {N} neurons, Hebbian weights, asynchronous updates")
        print(f"Stored    : digits {self.digits} ({self.mode}, {self.rule} rule); capacity limit ~{0.138*N:.1f} random patterns")
        print(f"Stable stored patterns: {self.stable_count()}/{len(self.digits)}")
        print(f"\n{'noise':>7}{'perfect recall':>16}{'right basin':>13}{'avg sweeps':>12}")
        for f in (0.05, 0.1, 0.2, 0.3, 0.4):
            ok = basin = 0; sw = []; T = 30
            for p, d in zip(self.net.P, self.digits):
                for _ in range(T):
                    x, s, _ = self.net.recall(self.noisy(p, f), self.rng); sw.append(s)
                    r = self.net.identify(x); ok += np.array_equal(x, p)
                    basin += (r[0] == "stored" and r[1] == d)
            n = T * len(self.digits)
            print(f"{f*100:>6.0f}%{ok/n*100:>15.1f}%{basin/n*100:>12.1f}%{np.mean(sw):>12.2f}")
        hit = tot = 0
        for d in self.digits:
            for i in self.rng.choice(np.where(self.y == d)[0], 30, replace=False):
                r = self.net.identify(self.net.recall(self.Xb[i], self.rng)[0]); tot += 1
                hit += (r[0] == "stored" and r[1] == d)
        print(f"\nReal unseen handwritten samples recalled to the right stored digit: {hit/tot*100:.1f}% ({hit}/{tot})")
        cnt = 0
        for _ in range(200):
            r = self.net.identify(self.net.recall(self.rng.choice([-1, 1], N), self.rng)[0]); cnt += r[0] != "stored"
        print(f"Random starts ending in a false memory (spurious or mirror): {cnt/2:.1f}%")


def main():
    X, y, source = load_data(sys.argv[1] if len(sys.argv) > 1 else None)
    print(f"[data] {len(y)} samples loaded.\n")
    app = App(X, y, source)
    ops = {"1": app.op1, "2": app.op2, "3": app.op3, "4": app.op4, "5": app.op5,
           "6": app.op6, "7": app.op7, "8": app.op8}
    while True:
        print("\n===== HOPFIELD DIGIT MEM SYSTEM =====")
        print("1. Store Handwritten Digit Patterns")
        print("2. Recall a Noisy Digit Pattern")
        print("3. Missing Pixel Reconstruction Challenge")
        print("4. Digit Corruption Tolerance Test")
        print("5. Hopfield Memory Capacity Challenge")
        print("6. False Memory Detector")
        print("7. Digit Confusion Analyzer")
        print("8. Hopfield Recall Performance Report")
        print("9. Exit")
        try:
            c = input("Choice: ").strip()
        except EOFError:
            break
        if c == "9": print("Goodbye."); break
        if c in ops: print(); ops[c]()
        else: print("Invalid choice.")


if __name__ == "__main__":
    main()

[data] Using scikit-learn bundled UCI optdigits (1797 samples).
[data] 1797 samples loaded.


===== HOPFIELD DIGIT MEM SYSTEM =====
1. Store Handwritten Digit Patterns
2. Recall a Noisy Digit Pattern
3. Missing Pixel Reconstruction Challenge
4. Digit Corruption Tolerance Test
5. Hopfield Memory Capacity Challenge
6. False Memory Detector
7. Digit Confusion Analyzer
8. Hopfield Recall Performance Report
9. Exit
Choice: 1

Digits to store (e.g. '2 6 7' or '0 1 2 3 4 5 6 7 8 9', default 2 6 7): 2
Pattern source: 1) class-average prototype (default)  2) one random sample each: 3
Learning rule: 1) Hebbian (default)  2) pseudo-inverse (comparison, handles correlated digits): 4

Stored 1 patterns by the Hebb rule  W = (1/64) sum x x^T  (prototype)

Digit 2  (energy -31.500)
   . . # # # . . .
   . . # # # . . .
   . . # . # . . .
   . . . . # . . .
   . . . # # . . .
   . . . # . . . .
   . . # # # # . .
   . . # # # # # .

Theoretical capacity of 64 neurons ~ 0.138*N = 8.8 random patterns
St

In [ ]:
import urllib.request
import zipfile
import os

# URL of the Dry Bean Dataset from UCI repository
url = 'https://archive.ics.uci.edu/static/public/602/dry+bean+dataset.zip'
zip_path = 'dry_bean_dataset.zip'

print("Downloading Dry Bean Dataset...")
urllib.request.urlretrieve(url, zip_path)

print("Extracting dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('dry_bean_temp')

# The dataset zip contains another zip internally: 'DryBeanDataset.zip'
inner_zip = 'dry_bean_temp/DryBeanDataset.zip'
if os.path.exists(inner_zip):
    with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extraction complete. 'DryBeanDataset' directory created.")
else:
    # Fallback to copy xlsx directly if layout changes
    for root, dirs, files in os.walk('dry_bean_temp'):
        for file in files:
            if file.endswith('.xlsx') or file.endswith('.csv'):
                os.rename(os.path.join(root, file), file)
    print("Extraction complete.")

# Clean up temp files
if os.path.exists(zip_path): os.remove(zip_path)
import shutil
if os.path.exists('dry_bean_temp'): shutil.rmtree('dry_bean_temp')


Extracting dataset...
Extraction complete.


In [ ]:
"""
Exercise 10: Adaptive Resonance Theory (Fuzzy ART) for Dynamic Category Formation and
             Novel Pattern Discovery using the Dry Bean Dataset

Network  : Fuzzy ART with complement coding.
           input  I = [a, 1 - a]  (a = min-max normalised features in [0, 1])
           choice T_j = |I ^ w_j| / (alpha + |w_j|)
           match      |I ^ w_j| / |I|  >= rho (vigilance)  -> resonance, learn  w_j = beta*(I ^ w_j) + (1-beta)*w_j
           otherwise   category j is reset, the next-best is tried;
           if every category is reset a NEW category is created (w = I).
Labels   : the bean 'Class' column is NEVER used for learning - only to measure purity afterwards.

Usage    : python exercise10_fuzzy_art_beans.py [path_to_dry_bean_file]
           Looks for Dry_Bean_Dataset.xlsx or Dry_Bean_Dataset.csv in the current folder
           (.xlsx needs openpyxl: pip install openpyxl).
Needs    : numpy, pandas
"""
import os
import sys
import numpy as np
import pandas as pd

RNG = np.random.default_rng(11)


# ----------------------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------------------
def load_beans(path=None):
    cands = ([path] if path else []) + ["Dry_Bean_Dataset.xlsx", "Dry_Bean_Dataset.csv",
                                        "DryBeanDataset/Dry_Bean_Dataset.xlsx", "dry_bean.csv"]
    f = next((p for p in cands if p and os.path.exists(p)), None)
    if f is None:
        raise SystemExit("Dry Bean dataset not found. Download Dry_Bean_Dataset.xlsx / .csv from\n"
                         "  https://archive.ics.uci.edu/dataset/602/dry+bean+dataset\n"
                         "and put it next to this script, or pass its path as an argument.")
    df = pd.read_excel(f) if f.lower().endswith((".xlsx", ".xls")) else pd.read_csv(f)
    df = df.dropna()
    ccol = next((c for c in df.columns if str(c).strip().lower() == "class"), df.columns[-1])
    names = [str(v) for v in df[ccol]]
    classes = sorted(set(names))
    y = np.array([classes.index(v) for v in names])
    Xdf = df.drop(columns=[ccol]).select_dtypes(include=[np.number])
    return Xdf.values.astype(float), y, classes, list(Xdf.columns), f


def minmax(X):
    lo, hi = X.min(axis=0), X.max(axis=0)
    span = np.where(hi - lo == 0, 1.0, hi - lo)
    return np.clip((X - lo) / span, 0, 1)


def complement(A):
    return np.hstack([A, 1.0 - A])


# ----------------------------------------------------------------------------------------
# Fuzzy ART
# ----------------------------------------------------------------------------------------
class FuzzyART:
    def __init__(self, dim2, n_classes, rho=0.75, alpha=0.001, beta=1.0):
        self.W = np.empty((0, dim2))
        self.lab = np.empty((0, n_classes), int)           # label tallies (analysis only)
        self.size = np.empty(0, int)
        self.rho, self.alpha, self.beta = rho, alpha, beta
        self.dim2, self.n_classes = dim2, n_classes
        self.resets_total = 0

    @property
    def K(self):
        return len(self.W)

    def _scores(self, I):
        m = np.minimum(I, self.W)
        inter = m.sum(axis=1)
        T = inter / (self.alpha + self.W.sum(axis=1))      # choice function
        match = inter / I.sum()                            # match (vigilance) function
        return T, match, m

    def _new(self, I, label):
        self.W = np.vstack([self.W, I])
        row = np.zeros((1, self.n_classes), int)
        if label is not None:
            row[0, label] = 1
        self.lab = np.vstack([self.lab, row])
        self.size = np.append(self.size, 1)
        return self.K - 1

    def present(self, I, label=None, learn=True, rho=None):
        """Returns (category, number_of_resets, created_new). category = -1 for a novel input when learn=False."""
        rho = self.rho if rho is None else rho
        if self.K == 0:
            return (self._new(I, label), 0, True) if learn else (-1, 0, True)
        T, match, m = self._scores(I)
        resets = 0
        for j in np.argsort(-T):
            if match[j] >= rho:                            # resonance
                if learn:
                    self.W[j] = self.beta * m[j] + (1 - self.beta) * self.W[j]
                    self.size[j] += 1
                    if label is not None:
                        self.lab[j, label] += 1
                self.resets_total += resets
                return int(j), resets, False
            resets += 1                                    # reset: vigilance test failed
        self.resets_total += resets
        if learn:
            return self._new(I, label), resets, True
        return -1, resets, True

    def analyze(self, I, top=5):
        T, match, _ = self._scores(I)
        order = np.argsort(-T)
        rows, winner = [], None
        for rank, j in enumerate(order):
            ok = match[j] >= self.rho
            if rank < top:
                rows.append((int(j), T[j], match[j], ok))
            if ok and winner is None:
                winner = int(j)
                if rank >= top:
                    rows.append((int(j), T[j], match[j], ok))
        return rows, winner

    def majority(self):
        return np.where(self.lab.sum(axis=1) > 0, self.lab.argmax(axis=1), -1)


def train(A_idx, C, y, rho, order=None, alpha=0.001, beta=1.0, net=None, checkpoints=None):
    """One incremental pass over the samples (in the given order). Returns net, assignment, history."""
    net = net or FuzzyART(C.shape[1], int(y.max()) + 1, rho, alpha, beta)
    net.rho = rho
    order = np.arange(len(A_idx)) if order is None else order
    assign = np.full(len(A_idx), -1)
    hist = []
    for n, i in enumerate(order, 1):
        j, _, _ = net.present(C[A_idx[i]], int(y[A_idx[i]]))
        assign[i] = j
        if checkpoints and n in checkpoints:
            hist.append((n, net.K))
    return net, assign, hist


def purity(assign, y):
    tot = 0
    for j in np.unique(assign):
        tot += np.bincount(y[assign == j]).max()
    return tot / len(y)


def ari(a, b):
    ua, ia = np.unique(a, return_inverse=True)
    ub, ib = np.unique(b, return_inverse=True)
    cont = np.zeros((len(ua), len(ub)))
    np.add.at(cont, (ia, ib), 1)
    comb = lambda x: x * (x - 1) / 2
    sc, sa, sb, n = comb(cont).sum(), comb(cont.sum(1)).sum(), comb(cont.sum(0)).sum(), comb(len(a))
    exp, mx = sa * sb / n, (sa + sb) / 2
    return (sc - exp) / (mx - exp) if mx != exp else 1.0


# ----------------------------------------------------------------------------------------
# State and helpers
# ----------------------------------------------------------------------------------------
class App:
    pass


app = App()


def ask(prompt, default, cast=int):
    s = input(f"{prompt} [default {default}]: ").strip()
    if not s:
        return default
    try:
        return cast(s)
    except ValueError:
        print("   invalid input, using default.")
        return default


def setup(path=None):
    X, y, classes, feats, f = load_beans(path)
    app.A = minmax(X)
    app.C = complement(app.A)
    app.y, app.classes, app.feats = y, classes, feats
    app.net, app.assign, app.rho = None, None, None
    print(f"Loaded '{f}': {len(y)} beans, {X.shape[1]} features (min-max normalised, complement-coded to "
          f"{app.C.shape[1]} inputs), {len(classes)} bean types: {', '.join(classes)}")


def subsample(n):
    n = min(n, len(app.y))
    return np.sort(RNG.choice(len(app.y), n, replace=False))


def need_net():
    if app.net is None:
        print("   (no trained ART network yet - training with rho = 0.75 on all beans)")
        train_default(0.75)
    return app.net


def train_default(rho):
    idx = np.arange(len(app.y))
    order = RNG.permutation(len(idx))
    cps = {int(v) for v in np.linspace(len(idx) / 10, len(idx), 10)}
    app.net, app.assign, hist = train(idx, app.C, app.y, rho, order, checkpoints=cps)
    app.rho, app.hist, app.idx = rho, hist, idx


# ----------------------------------------------------------------------------------------
# 1. Incremental category formation
# ----------------------------------------------------------------------------------------
def opt1():
    print("\n=== 1. Incremental Bean Category Formation ===")
    rho = ask("Vigilance parameter rho (0-1)", 0.75, float)
    train_default(rho)
    net = app.net
    print(f"\nCategories formed: {net.K}   (beans per category on average: {len(app.y) / net.K:.1f})")
    print("Category growth while beans arrive one at a time:")
    for n, k in app.hist:
        print(f"   after {n:>6} beans: {k:>4} categories  " + "#" * min(k, 60))
    print(f"\nPurity w.r.t. true bean type (not used in training): {purity(app.assign, app.y) * 100:.1f}%")
    print(f"Average resets per bean: {net.resets_total / len(app.y):.2f}")
    print_category_table(net, 10)


def print_category_table(net, top):
    maj = net.majority()
    order = np.argsort(-net.size)[:top]
    print(f"\n{'cat':>5}{'size':>7}  {'majority type':<12}{'purity':>8}")
    for j in order:
        tot = net.lab[j].sum()
        print(f"{j:>5}{net.size[j]:>7}  {app.classes[maj[j]]:<12}{net.lab[j].max() / max(tot, 1) * 100:>7.1f}%")
    if net.K > top:
        print(f"   ... {net.K - top} more categories")


# ----------------------------------------------------------------------------------------
# 2. Vigilance parameter challenge
# ----------------------------------------------------------------------------------------
def opt2():
    print("\n=== 2. Vigilance Parameter Challenge ===")
    n = ask("Beans to use", 4000)
    idx = subsample(n)
    order = RNG.permutation(len(idx))
    print(f"\n{'rho':>6}{'categories':>12}{'purity %':>10}{'avg resets':>12}{'beans/cat':>11}{'singletons':>12}")
    for rho in (0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95):
        net, assign, _ = train(idx, app.C, app.y, rho, order)
        pur = purity(assign, app.y[idx])
        print(f"{rho:>6.2f}{net.K:>12}{pur * 100:>10.1f}{net.resets_total / len(idx):>12.2f}"
              f"{len(idx) / net.K:>11.1f}{int((net.size == 1).sum()):>12}")
    print("\nLow vigilance -> few broad categories (mixed bean types, low purity). High vigilance -> many tight "
          "categories (high purity, but fragmentation and singletons).")


# ----------------------------------------------------------------------------------------
# 3. Resonance and reset analyzer
# ----------------------------------------------------------------------------------------
def opt3():
    print("\n=== 3. Resonance and Reset Analyzer ===")
    net = need_net()
    i = ask(f"Bean index (0-{len(app.y) - 1})", 0)
    i = i if 0 <= i < len(app.y) else 0
    rows, winner = net.analyze(app.C[i], top=6)
    print(f"\nBean {i} (true type {app.classes[app.y[i]]}), vigilance rho = {net.rho}")
    print(f"{'rank':>5}{'category':>10}{'choice T':>11}{'match':>9}   outcome")
    for r, (j, T, m, ok) in enumerate(rows, 1):
        print(f"{r:>5}{j:>10}{T:>11.4f}{m:>9.4f}   " + (f"RESONANCE (match >= {net.rho})" if ok else "RESET (match < rho)"))
    if winner is None:
        print("No category satisfies the vigilance test -> a NEW category would be created.")
    else:
        print(f"Bean joins category {winner} (majority type {app.classes[net.majority()[winner]]}).")
    # dataset-wide statistics
    sample = subsample(1500)
    res = np.array([net.present(app.C[k], learn=False)[1:] for k in sample])
    resets, novel = res[:, 0], res[:, 1].astype(bool)
    print(f"\nOver {len(sample)} beans (learning off): mean resets {resets.mean():.2f}, "
          f"{(resets == 0).mean() * 100:.1f}% resonate with their first choice, "
          f"{novel.mean() * 100:.1f}% would trigger a new category.")
# ----------------------------------------------------------------------------------------
# 4. Novel bean pattern detector
# ----------------------------------------------------------------------------------------
def opt4():
    print("\n=== 4. Novel Bean Pattern Detector ===")
    for k, c in enumerate(app.classes):
        print(f"   {k}. {c} ({int((app.y == k).sum())} beans)")
    h = ask("Hold-out (unseen) bean type number", 2)
    h = h if 0 <= h < len(app.classes) else 2
    rho = ask("Vigilance rho", 0.85, float)
    seen = np.where(app.y != h)[0]
    unseen = np.where(app.y == h)[0]
    RNG.shuffle(seen)
    tr, te_seen = seen[: int(0.7 * len(seen))], seen[int(0.7 * len(seen)):]
    net, _, _ = train(tr, app.C, app.y, rho, np.arange(len(tr)))
    det_seen = np.array([net.present(app.C[i], learn=False)[2] for i in te_seen])
    det_unseen = np.array([net.present(app.C[i], learn=False)[2] for i in unseen])
    best = lambda ids: np.array([net._scores(app.C[i])[1].max() for i in ids[:500]])
    print(f"\nTrained on {len(tr)} beans of {len(app.classes) - 1} types -> {net.K} categories.")
    print(f"Unseen type '{app.classes[h]}': flagged NOVEL for {det_unseen.mean() * 100:.1f}% of {len(unseen)} beans (detection rate)")
    print(f"Known types (held-out beans) flagged novel: {det_seen.mean() * 100:.1f}% of {len(te_seen)} (false alarms)")
    print(f"Mean best-match value: unseen {best(unseen).mean():.3f} vs known {best(te_seen).mean():.3f} (vigilance {rho})")
    print("\nNovel-bean flags by true type:")
    for k, c in enumerate(app.classes):
        ids = unseen if k == h else te_seen[app.y[te_seen] == k]
        if len(ids):
            r = np.mean([net.present(app.C[i], learn=False)[2] for i in ids])
            print(f"   {c:<10} {r * 100:>5.1f}%   {'(unseen)' if k == h else ''}")
    print("A higher vigilance detects more novelty but also raises false alarms on legitimate known beans.")


# ----------------------------------------------------------------------------------------
# 5. Category split and merge explorer
# ----------------------------------------------------------------------------------------
def opt5():
    print("\n=== 5. Category Split and Merge Explorer ===")
    net = need_net()
    maj = net.majority()
    purities = net.lab.max(axis=1) / np.maximum(net.lab.sum(axis=1), 1)
    print("-- SPLIT candidates: large categories mixing several bean types --")
    cands = [j for j in np.argsort(-net.size) if purities[j] < 0.9 and net.size[j] >= 20][:5]
    if not cands:
        print("   none (all sizeable categories are >= 90% pure).")
    for j in cands:
        members = np.where(app.assign == j)[0]
        mix = ", ".join(f"{app.classes[c]} {v}" for c, v in enumerate(net.lab[j]) if v)
        print(f"   category {j}: {net.size[j]} beans, purity {purities[j] * 100:.0f}%  [{mix}]")
        for r2 in (min(app.rho + 0.1, 0.99), min(app.rho + 0.2, 0.99)):
            sub, asg, _ = train(members, app.C, app.y, r2, np.arange(len(members)))
            print(f"      re-learned at rho={r2:.2f}: {sub.K} sub-categories, purity {purity(asg, app.y[members]) * 100:.0f}%")
    print("\n-- MERGE candidates: same-type categories with highly overlapping weights --")
    K = net.K
    sims = []
    sz = net.W.sum(axis=1)
    for a in range(K):
        inter = np.minimum(net.W[a], net.W[a + 1:]).sum(axis=1)
        for off, v in enumerate(inter):
            b = a + 1 + off
            s = v / min(sz[a], sz[b])
            if s >= 0.9 and maj[a] == maj[b] and maj[a] >= 0:
                sims.append((s, a, b))
    sims.sort(reverse=True)
    print(f"   {len(sims)} candidate pairs (similarity >= 0.90, same majority type)")
    for s, a, b in sims[:6]:
        print(f"   categories {a} & {b}: similarity {s:.3f}, sizes {net.size[a]} + {net.size[b]}, type {app.classes[maj[a]]}")
    if sims:
        merged = len({x for _, a, b in sims for x in (a, b)})
        print(f"   merging all candidate groups could remove roughly {max(1, merged // 2)} of {K} categories.")
print("\nSplit = raise vigilance locally; merge = collapse near-duplicate prototypes (lower vigilance).")


# ----------------------------------------------------------------------------------------
# 6. Catastrophic forgetting resistance test
# ----------------------------------------------------------------------------------------
class Softmax:
    """Online softmax regression trained with SGD - baseline that forgets."""
    def __init__(self, d, k, lr=0.1, seed=0):
        self.W = np.zeros((d + 1, k))
        self.lr, self.rng = lr, np.random.default_rng(seed)

    def fit(self, X, y, epochs=30):
        Xa = np.hstack([X, np.ones((len(X), 1))])
        for _ in range(epochs):
            for i in self.rng.permutation(len(X)):
                z = Xa[i] @ self.W
                p = np.exp(z - z.max())
                p /= p.sum()
                p[y[i]] -= 1
                self.W -= self.lr * np.outer(Xa[i], p)

    def predict(self, X):
        return np.argmax(np.hstack([X, np.ones((len(X), 1))]) @ self.W, axis=1)


def art_predict(net, ids):
    maj = net.majority()
    out = []
    for i in ids:
        j, _, _ = net.present(app.C[i], learn=False)
        out.append(maj[j] if j >= 0 else -1)
    return np.array(out)


def opt6():
    print("\n=== 6. Catastrophic Forgetting Resistance Test ===")
    K = len(app.classes)
    for k, c in enumerate(app.classes):
        print(f"   {k}. {c}")
    s = input(f"Task A bean types, comma separated [default 0,1,2]: ").strip() or "0,1,2"
    try:
        A = sorted({int(t) for t in s.split(",") if t.strip() != ""})
        assert A and all(0 <= a < K for a in A) and len(A) < K
    except Exception:
        print("   invalid list, using 0,1,2.")
        A = [0, 1, 2]
    B = [k for k in range(K) if k not in A]
    rho = ask("Vigilance rho", 0.85, float)
    idx = subsample(6000)
    RNG.shuffle(idx)
    cut = int(0.7 * len(idx))
    tr, te = idx[:cut], idx[cut:]
    trA, trB = tr[np.isin(app.y[tr], A)], tr[np.isin(app.y[tr], B)]
    teA, teB = te[np.isin(app.y[te], A)], te[np.isin(app.y[te], B)]
    print(f"\nTask A = {[app.classes[a] for a in A]} ({len(trA)} train / {len(teA)} test)")
    print(f"Task B = {[app.classes[b] for b in B]} ({len(trB)} train / {len(teB)} test)\n")
    # ART
    net, _, _ = train(trA, app.C, app.y, rho, np.arange(len(trA)))
    kA = net.K
    a1 = (art_predict(net, teA) == app.y[teA]).mean() * 100
    train(trB, app.C, app.y, rho, np.arange(len(trB)), net=net)
    a2 = (art_predict(net, teA) == app.y[teA]).mean() * 100
    b2 = (art_predict(net, teB) == app.y[teB]).mean() * 100
    # baseline
    sm = Softmax(app.A.shape[1], K)
    sm.fit(app.A[trA], app.y[trA])
    s1 = (sm.predict(app.A[teA]) == app.y[teA]).mean() * 100
    sm.fit(app.A[trB], app.y[trB])
    s2 = (sm.predict(app.A[teA]) == app.y[teA]).mean() * 100
    sb = (sm.predict(app.A[teB]) == app.y[teB]).mean() * 100
    print(f"{'model':<26}{'Task A after A':>16}{'Task A after B':>16}{'Task B after B':>16}{'forgetting of A':>17}")
    print(f"{'Fuzzy ART (rho=' + str(rho) + ')':<26}{a1:>15.1f}%{a2:>15.1f}%{b2:>15.1f}%{a1 - a2:>16.1f}")
    print(f"{'Softmax SGD (baseline)':<26}{s1:>15.1f}%{s2:>15.1f}%{sb:>15.1f}%{s1 - s2:>16.1f}")
    print(f"\nART grew from {kA} to {net.K} categories while learning Task B; old categories were left untouched "
          "unless a Task B bean genuinely matched them (vigilance protects them).")
    print("The baseline overwrites shared weights, so its Task A accuracy collapses after training on Task B only.")
# ----------------------------------------------------------------------------------------
# 7. Input order sensitivity
# ----------------------------------------------------------------------------------------
def opt7():
    print("\n=== 7. Input Order Sensitivity Challenge ===")
    rho = ask("Vigilance rho", 0.8, float)
    runs = ask("Random orderings", 8)
    idx = subsample(3000)
    y = app.y[idx]
    results, assigns = [], []
    for r in range(runs):
        order = RNG.permutation(len(idx))
        net, asg, _ = train(idx, app.C, app.y, rho, order)
        results.append((f"random #{r + 1}", net.K, purity(asg, y)))
        assigns.append(asg)
    named = {"sorted by bean type": np.argsort(y, kind="stable"),
             "reverse by bean type": np.argsort(-y, kind="stable"),
             "ascending Area feature": np.argsort(app.A[idx][:, 0])}
    for name, order in named.items():
        net, asg, _ = train(idx, app.C, app.y, rho, order)
        results.append((name, net.K, purity(asg, y)))
        assigns.append(asg)
    print(f"\n{'ordering':<24}{'categories':>11}{'purity %':>10}{'ARI vs random #1':>18}")
    for (n, k, p), a in zip(results, assigns):
        print(f"{n:<24}{k:>11}{p * 100:>10.1f}{ari(assigns[0], a):>18.3f}")
    ks = np.array([k for n, k, p in results[:runs]])
    print(f"\nRandom orderings: categories mean {ks.mean():.1f} +/- {ks.std():.1f} (min {ks.min()}, max {ks.max()}).")
    pair = [ari(assigns[a], assigns[b]) for a in range(runs) for b in range(a + 1, runs)]
    if pair:
        print(f"Mean pairwise Adjusted Rand Index between random orderings: {np.mean(pair):.3f} (1.0 = identical partitions)")
    print("ART is order dependent: prototypes depend on which beans arrive first, though purity usually stays similar.")


# ----------------------------------------------------------------------------------------
# 8. Report
# ----------------------------------------------------------------------------------------
def opt8():
    print("\n=== 8. ART Category Formation Report ===")
    net = need_net()
    maj = net.majority()
    print(f"Vigilance rho = {net.rho}, alpha = {net.alpha}, beta = {net.beta} (fast learning), "
          f"inputs = {app.C.shape[1]} (complement coded)")
    print(f"Beans: {len(app.y)}   Categories: {net.K}   Compression: {len(app.y) / net.K:.1f} beans/category")
    print(f"Overall purity: {purity(app.assign, app.y) * 100:.1f}%   Average resets per bean: {net.resets_total / len(app.y):.2f}")
    print(f"Singleton categories: {int((net.size == 1).sum())}   Largest category: {net.size.max()} beans")
    print(f"\n{'bean type':<12}{'beans':>7}{'categories':>12}{'in dominant cat %':>19}")
    for k, c in enumerate(app.classes):
        m = app.y == k
        cats, cnt = np.unique(app.assign[m], return_counts=True)
        print(f"{c:<12}{int(m.sum()):>7}{len(cats):>12}{cnt.max() / m.sum() * 100:>18.1f}%")
    print_category_table(net, 12)
    print("\nCategory prototype ranges for the largest category (original min-max scale 0-1):")
    j = int(np.argmax(net.size))
    d = app.A.shape[1]
    lo, hi = net.W[j, :d], 1 - net.W[j, d:]
    for f, a, b in list(zip(app.feats, lo, hi))[:6]:
        print(f"   {f:<18} [{a:.3f}, {b:.3f}]")
    print("   ...")


MENU = """
============ Fuzzy ART Dry Bean System ============
 1. Incremental Bean Category Formation
 2. Vigilance Parameter Challenge
 3. Resonance and Reset Analyzer
 4. Novel Bean Pattern Detector
 5. Category Split and Merge Explorer
 6. Catastrophic Forgetting Resistance Test
 7. Input Order Sensitivity Challenge
 8. ART Category Formation Report
 9. Exit
===================================================="""
ACTIONS = {"1": opt1, "2": opt2, "3": opt3, "4": opt4, "5": opt5, "6": opt6, "7": opt7, "8": opt8}
def main():
    setup(sys.argv[1] if len(sys.argv) > 1 else None)
    while True:
        print(MENU)
        try:
            c = input("Enter your choice (1-9): ").strip()
        except EOFError:
            break
        if c == "9":
            print("Goodbye!")
            break
        fn = ACTIONS.get(c)
        if fn is None:
            print("Invalid choice, please enter a number from 1 to 9.")
            continue
        fn()


if __name__ == "__main__":
    main()


Split = raise vigilance locally; merge = collapse near-duplicate prototypes (lower vigilance).
Loaded 'Dry_Bean_Dataset.xlsx': 13611 beans, 16 features (min-max normalised, complement-coded to 32 inputs), 7 bean types: BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ, SEKER, SIRA

============ Fuzzy ART Dry Bean System ============
 1. Incremental Bean Category Formation
 2. Vigilance Parameter Challenge
 3. Resonance and Reset Analyzer
 4. Novel Bean Pattern Detector
 5. Category Split and Merge Explorer
 6. Catastrophic Forgetting Resistance Test
 7. Input Order Sensitivity Challenge
 8. ART Category Formation Report
 9. Exit

=== 1. Incremental Bean Category Formation ===

Categories formed: 13611   (beans per category on average: 1.0)
Category growth while beans arrive one at a time:
   after   1361 beans: 1361 categories  ############################################################
   after   2722 beans: 2722 categories  ############################################################
   afte

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# We will use two representative features to visualize the 2D category bounding boxes
# Feature 0: Area, Feature 1: Perimeter
feat_idx = [0, 1]
x_label = app.feats[feat_idx[0]]
y_label = app.feats[feat_idx[1]]

# Extract 2D normalized data for visualization
coord_data = app.A[:, feat_idx]
complement_data = np.hstack([coord_data, 1.0 - coord_data])

# Take a subset of samples for quick training/plotting
plot_samples = subsample(1500)
sub_C = complement_data[plot_samples]
sub_y = app.y[plot_samples]

vigilance_levels = [0.65, 0.80, 0.95]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharex=True, sharey=True)

for ax, rho in zip(axes, vigilance_levels):
    # Initialize Fuzzy ART for 2D inputs (+ complement = 4 inputs)
    net = FuzzyART(dim2=4, n_classes=len(app.classes), rho=rho, beta=1.0)

    # Train the network on the 2D subspace
    for i in range(len(plot_samples)):
        net.present(sub_C[i], label=int(sub_y[i]))

    # Plot the normalized data points
    ax.scatter(coord_data[plot_samples, 0], coord_data[plot_samples, 1], c=sub_y, cmap='tab10', alpha=0.3, s=10)

    # Draw a bounding box for each category template
    # For 2D complement coding, the weight vector is [u0, u1, 1-v0, 1-v1]
    # The bounding box is defined by interval [u0, v0] on x-axis and [u1, v1] on y-axis
    for j in range(net.K):
        u0, u1 = net.W[j, 0], net.W[j, 1]
        v0, v1 = 1.0 - net.W[j, 2], 1.0 - net.W[j, 3]

        width = max(v0 - u0, 0.005)  # add a tiny minimum width so singletons are visible
        height = max(v1 - u1, 0.005)

        # Create the rectangle patch
        rect = patches.Rectangle(
            (u0, u1), width, height,
            linewidth=1.5, edgecolor='black', facecolor='none', alpha=0.8
        )
        ax.add_patch(rect)

    ax.set_title(f"Vigilance ($\rho$) = {rho}\nCategories (K) = {net.K}", fontsize=12)
    ax.set_xlabel(f"Normalized {x_label}")
    if ax == axes[0]:
        ax.set_ylabel(f"Normalized {y_label}")
    ax.grid(True, linestyle='--', alpha=0.5)

plt.suptitle("Fuzzy ART Category Boundaries (Bounding Boxes) vs. Vigilance", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()